In [7]:
# Installation des packages et récupération des données
import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install matplotlib tqdm seaborn scikit-learn

!pip install opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 1.5 MB/s  0:00:06 eta 0:00:020m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 3.2 MB/s  0:00:10m0:00:0100:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [seaborn]m4/5 [seaborn]earn]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 78.5 MB/s  0:00:00 eta 0:00:01


In [11]:
#importation des modules
import cv2 
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import os
import shutil
from tqdm import tqdm
import pandas as pd

In [12]:
# Chargement et filtrage des annotations
df = pd.read_csv("../VinDr/breast-level_annotations.csv")
df = df[df["view_position"] == "MLO"]
df = df[df["breast_birads"].isin(["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"])]
df["label"] = (df["breast_birads"] == "BI-RADS 1").astype(int)  # 1=normal, 0=cancer

In [15]:
# Restructuration VinDr
import os
import shutil

images_dir = "../VinDr/images_png"

destination_normal = "../VinDr_structure/normal"
destination_cancer = "../VinDr_structure/cancer_benign"
os.makedirs(destination_normal, exist_ok=True)
os.makedirs(destination_cancer, exist_ok=True)

for _, row in tqdm(df.iterrows(), total=len(df)):
    img_path = os.path.join(images_dir, row["study_id"], row["image_id"] + ".png")
    if not os.path.exists(img_path):
        continue
    if row["label"] == 1:
        shutil.copy2(img_path, destination_normal)
    else:
        shutil.copy2(img_path, destination_cancer)

# Vérification
for classe in ["normal", "cancer_benign"]:
    nb = len(os.listdir(f"../VinDr_transforme/{classe}"))
    print(f"  - {classe}: {nb} images")

# Sauvegarde sur S3
!mc cp -r ../VinDr_structure/ s3/dimitri/stat_app/vindr_restructure/

100%|██████████| 9999/9999 [00:12<00:00, 773.08it/s] 


  - normal: 0 images
  - cancer_benign: 0 images
]11;?\mc: Configuration written to `/home/onyxia/.mc/config.json`. Please update your access credentials.
mc: Successfully created `/home/onyxia/.mc/share`.
mc: Initialized share uploads `/home/onyxia/.mc/share/uploads.json` file.
mc: Initialized share downloads `/home/onyxia/.mc/share/downloads.json` file.
...cdc4dc.png: 4.20 GiB / 4.20 GiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 240.32 MiB/s 17s